# Seasonal Agriculture Performance Analysis

## Major Project – Data Analytics

### Project Introduction
Agriculture is strongly affected by seasonal conditions, farming practices, resource availability, and economic factors. A farm may produce a high quantity of crops but still experience low profitability because of excessive input costs or inefficient water use.

This project studies **4,000 agricultural records across Kharif, Rabi, and Zaid seasons**. The analysis focuses on understanding how climate, irrigation, fertilizer use, crop yield, water efficiency, and profit interact with each other.

**Important:** This notebook is designed as an original analysis workflow. It follows the project theme and expected outcomes, but uses a different analytical structure, derived metrics, and investigation questions.

---

## 1. Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability and market conditions. 

As a result, agricultural performance may differ from one season to another.
However, raw agricultural data does not clearly explain how agricultural performance changes across seasons or what patterns can be observed in different seasonal conditions.

The problem is to analyze the given agricultural dataset and investigate seasonal differences in agricultural performance by identifying meaningful patterns, trends, relationships and variations within the available data. 
 
---

## 2. Project Objectives

The primary objective of this project is to analyze agricultural data from different seasons and identify meaningful patterns, trends, relationships and differences in agricultural performance. 

Students should independently:
- Explore and understand the dataset.
- Clean and prepare the data for analysis.
- Examine how agricultural performance varies across seasons.
- Identify important seasonal patterns and trends.
- Investigate relationships between seasonal conditions and agricultural outcomes.
- Compare relevant groups within different seasons.
- Identify significant differences or unusual patterns.
- Apply appropriate statistical and visualization techniques.
- Interpret findings based on evidence from the dataset.
- Develop meaningful conclusions and data-driven recommendations.

---

## 3. Key Analytical Questions

- Which season performs best overall?
- Does high rainfall always result in high crop yield?
- Which characteristics change between seasons?
- Which irrigation method provides the best balance between water use and productivity?
- Do farms using more fertilizer always earn more profit?
- Which crops provide the strongest yield-profit combination?
- What factors are associated with loss-making farms?
- How does farm size influence profitability?
- Which season requires the greatest resource-management attention?
- How does agricultural performance vary across seasons?
- What differences exist between agricultural activities in different seasons?
- Are some seasonal patterns consistent across different regions or categories?
- What conclusions can reasonably be drawn from the available data? 
  


In [ ]:
# 4. Library Imports and Setup
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Libraries loaded successfully.")

In [ ]:
# 5. Load Dataset
# Keep the CSV file in the same folder as this notebook.
file_path = "seasonal_agriculture_performance_dataset.csv"

df = pd.read_csv(file_path)

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 6. Dataset Understanding

The dataset includes:

- **Farm details:** Farm ID, State, District, Farm Area
- **Crop details:** Crop and Season
- **Environmental factors:** Rainfall, Temperature, Humidity, Sunlight
- **Soil conditions:** Soil pH and Soil Moisture
- **Farm inputs:** NPK nutrients, Fertilizer, Pesticide
- **Resource management:** Irrigation Method and Water Used
- **Agricultural outcomes:** Yield and Production
- **Economic outcomes:** Cost, Revenue, and Profit
- **Risk indicator:** Disease and Pest Risk


In [ ]:
# 6.1 Basic Structural Inspection
print("Column Names:")
print(df.columns.tolist())

print("\nData Types and Non-Null Counts:")
df.info()

print("\nRandom Sample:")
display(df.sample(5, random_state=42))

In [ ]:
# 6.2 Missing Values and Duplicate Records
missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

display(missing[missing["Missing_Count"] > 0])

print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Visualize missing values only when they exist
missing_nonzero = missing[missing["Missing_Count"] > 0]

if not missing_nonzero.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(
        data=missing_nonzero.reset_index(),
        x="Missing_Count",
        y="index",
        hue="index",
        legend=False
    )
    plt.title("Missing Values by Feature")
    plt.xlabel("Missing Records")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 7. Data Cleaning Strategy

Instead of using one global average for every missing value, this project uses **context-based median imputation**:

- Rainfall is filled using the median for the corresponding **Season**.
- Soil moisture is filled using **Season + Irrigation Method**.
- Yield is filled using **Crop + Season**.

This approach better preserves the natural differences between agricultural groups.


In [ ]:
# 7.1 Context-Based Missing Value Treatment
df_clean = df.copy()

df_clean["Rainfall_mm"] = (
    df_clean.groupby("Season")["Rainfall_mm"]
    .transform(lambda x: x.fillna(x.median()))
)

df_clean["Soil_Moisture_pct"] = (
    df_clean.groupby(["Season", "Irrigation_Method"])["Soil_Moisture_pct"]
    .transform(lambda x: x.fillna(x.median()))
)

df_clean["Yield_Tonnes_Ha"] = (
    df_clean.groupby(["Crop", "Season"])["Yield_Tonnes_Ha"]
    .transform(lambda x: x.fillna(x.median()))
)

print("Remaining missing values:", df_clean.isnull().sum().sum())

## 8. Feature Engineering

To make the project more original and decision-oriented, new analytical features are created:

- **Cost per tonne** – production cost relative to output.
- **Profit margin** – percentage of revenue retained as profit.
- **Revenue per hectare** – economic output relative to farm size.
- **Water use per hectare** – resource intensity.
- **Fertilizer efficiency** – yield obtained per unit of fertilizer.
- **Farm size category** – Small, Medium, and Large farms.
- **Performance score** – a combined normalized score based on yield, profit, and water efficiency.


In [ ]:
# 8.1 Create Derived Features
df_clean["Cost_Per_Tonne"] = (
    df_clean["Total_Cost_INR"] / df_clean["Production_Tonnes"].replace(0, np.nan)
)

df_clean["Profit_Margin_pct"] = (
    (df_clean["Profit_INR"] / df_clean["Revenue_INR"].replace(0, np.nan)) * 100
)

df_clean["Revenue_Per_Hectare"] = (
    df_clean["Revenue_INR"] / df_clean["Farm_Area_Hectares"].replace(0, np.nan)
)

df_clean["Water_Per_Hectare"] = (
    df_clean["Water_Used_m3"] / df_clean["Farm_Area_Hectares"].replace(0, np.nan)
)

df_clean["Fertilizer_Efficiency"] = (
    df_clean["Yield_Tonnes_Ha"] / df_clean["Fertilizer_kg_ha"].replace(0, np.nan)
)

df_clean["Farm_Size_Category"] = pd.cut(
    df_clean["Farm_Area_Hectares"],
    bins=[0, 5, 10, np.inf],
    labels=["Small", "Medium", "Large"]
)

df_clean["Profit_Status"] = np.where(
    df_clean["Profit_INR"] >= 0,
    "Profitable",
    "Loss-Making"
)

display(df_clean[[
    "Cost_Per_Tonne",
    "Profit_Margin_pct",
    "Revenue_Per_Hectare",
    "Water_Per_Hectare",
    "Fertilizer_Efficiency",
    "Farm_Size_Category",
    "Profit_Status"
]].head())

In [ ]:
# 8.2 Overall Descriptive Statistics
numeric_cols = df_clean.select_dtypes(include=np.number).columns

summary = df_clean[numeric_cols].describe().T
summary["Range"] = summary["max"] - summary["min"]
summary["IQR"] = summary["75%"] - summary["25%"]

display(summary.round(2))

# 9. Exploratory Data Analysis

In [ ]:
# 9.1 Distribution of Records Across Seasons
season_counts = df_clean["Season"].value_counts()

plt.figure(figsize=(7, 5))
sns.barplot(
    x=season_counts.index,
    y=season_counts.values,
    hue=season_counts.index,
    legend=False
)
plt.title("Number of Agricultural Records by Season", weight="bold")
plt.xlabel("Season")
plt.ylabel("Number of Farms")
plt.tight_layout()
plt.show()

In [ ]:
# 9.2 Climate Profile by Season
climate_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day"
]

seasonal_climate = df_clean.groupby("Season")[climate_cols].mean().round(2)
display(seasonal_climate)

seasonal_climate.plot(kind="bar", figsize=(12, 6))
plt.title("Average Seasonal Climate Conditions", weight="bold")
plt.xlabel("Season")
plt.ylabel("Average Value")
plt.xticks(rotation=0)
plt.legend(title="Indicator")
plt.tight_layout()
plt.show()

In [ ]:
# 9.3 Yield and Profit Comparison Across Seasons
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(
    data=df_clean,
    x="Season",
    y="Yield_Tonnes_Ha",
    ax=axes[0]
)
axes[0].set_title("Yield Distribution Across Seasons", weight="bold")

sns.boxplot(
    data=df_clean,
    x="Season",
    y="Profit_INR",
    ax=axes[1]
)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Profit Distribution Across Seasons", weight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# 9.4 Crop Performance: Yield vs Profit
crop_performance = (
    df_clean.groupby("Crop")
    .agg(
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Revenue=("Revenue_INR", "mean"),
        Farms=("Farm_ID", "count")
    )
    .reset_index()
)

display(crop_performance.sort_values("Avg_Profit", ascending=False).round(2))

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=crop_performance,
    x="Avg_Yield",
    y="Avg_Profit",
    size="Farms",
    hue="Crop",
    sizes=(100, 800)
)

for _, row in crop_performance.iterrows():
    plt.text(row["Avg_Yield"], row["Avg_Profit"], row["Crop"], fontsize=9)

plt.axhline(0, linestyle="--")
plt.title("Crop Yield-Profit Positioning", weight="bold")
plt.xlabel("Average Yield (Tonnes/Ha)")
plt.ylabel("Average Profit (INR)")
plt.tight_layout()
plt.show()

## 10. Irrigation and Resource Efficiency Analysis

In [ ]:
# 10.1 Irrigation Method Comparison
irrigation_summary = (
    df_clean.groupby("Irrigation_Method")
    .agg(
        Avg_Water_Used=("Water_Used_m3", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Pest_Risk=("Disease_Pest_Risk_pct", "mean")
    )
    .round(2)
)

display(irrigation_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=irrigation_summary.reset_index(),
    x="Irrigation_Method",
    y="Avg_Water_Efficiency",
    ax=axes[0]
)
axes[0].set_title("Water Efficiency by Irrigation Method", weight="bold")

sns.barplot(
    data=irrigation_summary.reset_index(),
    x="Irrigation_Method",
    y="Avg_Water_Used",
    ax=axes[1]
)
axes[1].set_title("Average Water Consumption by Irrigation Method", weight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# 10.2 Irrigation Method Performance Across Seasons
irrigation_season = (
    df_clean.groupby(["Season", "Irrigation_Method"])
    .agg(
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
    )
    .reset_index()
)

display(irrigation_season.round(2))

plt.figure(figsize=(11, 5))
sns.barplot(
    data=irrigation_season,
    x="Irrigation_Method",
    y="Avg_Water_Efficiency",
    hue="Season"
)
plt.title("Seasonal Water Efficiency by Irrigation Method", weight="bold")
plt.tight_layout()
plt.show()

## 11. Relationship Analysis

In [ ]:
# 11.1 Correlation Heatmap
analysis_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Soil_Moisture_pct",
    "Fertilizer_kg_ha",
    "Water_Used_m3",
    "Yield_Tonnes_Ha",
    "Revenue_INR",
    "Profit_INR",
    "Disease_Pest_Risk_pct",
    "Profit_Margin_pct"
]

plt.figure(figsize=(12, 8))
sns.heatmap(
    df_clean[analysis_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Relationship Between Agricultural Performance Indicators", weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 11.2 Rainfall vs Yield by Season
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_clean,
    x="Rainfall_mm",
    y="Yield_Tonnes_Ha",
    hue="Season",
    alpha=0.6
)
plt.title("Rainfall and Crop Yield Relationship", weight="bold")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

In [ ]:
# 11.3 Farm Size and Profitability
size_summary = (
    df_clean.groupby("Farm_Size_Category", observed=False)
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Profit_Margin=("Profit_Margin_pct", "mean")
    )
    .reset_index()
)

display(size_summary.round(2))

plt.figure(figsize=(9, 5))
sns.barplot(
    data=size_summary,
    x="Farm_Size_Category",
    y="Avg_Profit",
    hue="Farm_Size_Category",
    legend=False
)
plt.axhline(0, linestyle="--")
plt.title("Average Profit by Farm Size Category", weight="bold")
plt.tight_layout()
plt.show()

# 12. Student-Designed Advanced Investigations

The following investigations are intentionally different from a basic season-by-season analysis and add a stronger decision-making component to the project.


## Investigation 1: Does More Fertilizer Mean Better Results?

Farms are divided into fertilizer usage groups. The objective is to compare yield and profit across fertilizer intensity levels and identify possible diminishing returns.


In [ ]:
# 12.1 Fertilizer Intensity Analysis
df_clean["Fertilizer_Level"] = pd.qcut(
    df_clean["Fertilizer_kg_ha"],
    q=4,
    labels=["Low", "Moderate", "High", "Very High"]
)

fertilizer_analysis = (
    df_clean.groupby("Fertilizer_Level", observed=False)
    .agg(
        Avg_Fertilizer=("Fertilizer_kg_ha", "mean"),
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Cost=("Total_Cost_INR", "mean")
    )
    .reset_index()
)

display(fertilizer_analysis.round(2))

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=fertilizer_analysis,
    x="Fertilizer_Level",
    y="Avg_Yield",
    marker="o",
    label="Average Yield"
)
sns.lineplot(
    data=fertilizer_analysis,
    x="Fertilizer_Level",
    y="Avg_Profit",
    marker="o",
    label="Average Profit"
)
plt.title("Yield and Profit Across Fertilizer Intensity Levels", weight="bold")
plt.tight_layout()
plt.show()

## Investigation 2: What Makes a Farm Loss-Making?

This analysis compares profitable and loss-making farms to identify differences in yield, cost, water use, and pest risk.


In [ ]:
# 12.2 Loss-Making Farm Profile
profit_profile = (
    df_clean.groupby("Profit_Status")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Cost=("Total_Cost_INR", "mean"),
        Avg_Revenue=("Revenue_INR", "mean"),
        Avg_Water_Used=("Water_Used_m3", "mean"),
        Avg_Pest_Risk=("Disease_Pest_Risk_pct", "mean")
    )
    .reset_index()
)

display(profit_profile.round(2))

profile_melted = profit_profile.melt(
    id_vars="Profit_Status",
    value_vars=[
        "Avg_Yield",
        "Avg_Cost",
        "Avg_Revenue",
        "Avg_Water_Used",
        "Avg_Pest_Risk"
    ],
    var_name="Metric",
    value_name="Value"
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=profile_melted,
    x="Metric",
    y="Value",
    hue="Profit_Status"
)
plt.title("Comparison of Profitable and Loss-Making Farms", weight="bold")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## Investigation 3: Seasonal Performance Score

A normalized score is created to compare seasons using three positive performance indicators:

- Yield
- Profit
- Water Efficiency

This gives a broader comparison than using only one variable.


In [ ]:
# 12.3 Create a Normalized Performance Score
score_cols = [
    "Yield_Tonnes_Ha",
    "Profit_INR",
    "Water_Efficiency_t_per_1000m3"
]

for col in score_cols:
    min_val = df_clean[col].min()
    max_val = df_clean[col].max()
    df_clean[col + "_Norm"] = (
        (df_clean[col] - min_val) / (max_val - min_val)
    )

df_clean["Performance_Score"] = (
    0.35 * df_clean["Yield_Tonnes_Ha_Norm"]
    + 0.40 * df_clean["Profit_INR_Norm"]
    + 0.25 * df_clean["Water_Efficiency_t_per_1000m3_Norm"]
)

season_score = (
    df_clean.groupby("Season")
    .agg(
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
        Performance_Score=("Performance_Score", "mean")
    )
    .sort_values("Performance_Score", ascending=False)
)

display(season_score.round(4))

plt.figure(figsize=(8, 5))
sns.barplot(
    data=season_score.reset_index(),
    x="Season",
    y="Performance_Score",
    hue="Season",
    legend=False
)
plt.title("Overall Seasonal Performance Score", weight="bold")
plt.ylabel("Average Normalized Score")
plt.tight_layout()
plt.show()

# 13. Seasonal Performance Dashboard Table

In [ ]:
# 13.1 Combined Seasonal Dashboard
season_dashboard = (
    df_clean.groupby("Season")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Rainfall=("Rainfall_mm", "mean"),
        Avg_Temperature=("Avg_Temperature_C", "mean"),
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Total_Production=("Production_Tonnes", "sum"),
        Avg_Profit=("Profit_INR", "mean"),
        Total_Profit=("Profit_INR", "sum"),
        Avg_Water_Used=("Water_Used_m3", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
        Avg_Profit_Margin=("Profit_Margin_pct", "mean"),
        Performance_Score=("Performance_Score", "mean")
    )
    .round(2)
)

display(season_dashboard)

# 14. Automated Insight Generation

The notebook calculates key comparisons directly from the dataset. This makes the final interpretation evidence-driven rather than based on fixed assumptions.


In [ ]:
# 14.1 Generate Data-Based Highlights
best_yield_season = season_dashboard["Avg_Yield"].idxmax()
best_profit_season = season_dashboard["Avg_Profit"].idxmax()
best_efficiency_season = season_dashboard["Avg_Water_Efficiency"].idxmax()
best_score_season = season_dashboard["Performance_Score"].idxmax()

best_crop_profit = crop_performance.loc[
    crop_performance["Avg_Profit"].idxmax(), "Crop"
]

best_irrigation = irrigation_summary["Avg_Water_Efficiency"].idxmax()

print("KEY DATA-BASED HIGHLIGHTS")
print("-" * 45)
print(f"Highest average yield season: {best_yield_season}")
print(f"Highest average profit season: {best_profit_season}")
print(f"Highest water efficiency season: {best_efficiency_season}")
print(f"Best overall performance score: {best_score_season}")
print(f"Crop with highest average profit: {best_crop_profit}")
print(f"Most water-efficient irrigation method: {best_irrigation}")

# 15. Recommendations

Based on the analysis, recommendations should focus on the patterns observed in the dataset:

1. **Promote efficient irrigation methods** where they demonstrate lower water use and stronger water productivity.
2. **Monitor fertilizer intensity** because higher input usage may not always create proportional improvements in yield or profit.
3. **Use seasonal planning** to align crop and resource decisions with changing climate conditions.
4. **Investigate loss-making farms** by focusing on cost structure, revenue, water use, and agricultural risk.
5. **Compare crop decisions using both yield and profit**, because high production alone does not guarantee economic success.
6. **Use farm-size-specific planning**, since resource requirements and profitability may vary across small, medium, and large farms.

---

# 16. Limitations

- The analysis is based only on the available dataset.
- Results represent observed relationships and do not prove direct causation.
- The dataset does not provide multi-year historical trends for long-term forecasting.
- Market conditions can change over time.
- N, P, and K interactions can be explored further in future work.

---

# 17. Conclusion

This project analyzed seasonal agricultural performance using data cleaning, exploratory analysis, feature engineering, resource-efficiency analysis, economic analysis, and three advanced investigations.

A key strength of this notebook is that it does not evaluate agriculture using only crop yield. It combines **productivity, profitability, and resource efficiency** to provide a broader understanding of farm performance.

The analysis can support better seasonal planning by identifying relationships between environmental conditions, agricultural inputs, irrigation practices, crop outcomes, and financial performance.

---

## Future Scope

Possible future extensions include:

- Machine learning for yield or profit prediction
- Regional comparison dashboards
- Weather-based agricultural forecasting
- Interactive Streamlit or Power BI dashboard
- Crop recommendation based on seasonal conditions
